# Real Estate Price Valuation with RAG + Q&A
This notebook demonstrates a Retrieval-Augmented Generation (RAG) + Q&A system for analyzing real estate price data using property descriptions, PDF market reports, and CSV sales data.

In [ ]:
# Install dependencies
!pip install gradio langchain pypdf pandas tiktoken faiss-cpu sentence-transformers transformers


In [ ]:
import os
import gradio as gr
import pandas as pd
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFaceHub
from tempfile import NamedTemporaryFile
import requests


In [ ]:
# Load and split documents from CSV or PDF
def load_documents(file_obj):
    docs = []
    ext = os.path.splitext(file_obj.name)[-1].lower()
    if ext == ".pdf":
        loader = PyPDFLoader(file_obj.name)
        docs.extend(loader.load())
    elif ext == ".csv":
        df = pd.read_csv(file_obj.name)
        for _, row in df.iterrows():
            content = str(row.to_dict())
            docs.append(content)
    return docs


In [ ]:
# Create embeddings and QA chain
def create_qa_chain(docs):
    splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    texts = splitter.create_documents(docs)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    db = FAISS.from_documents(texts, embeddings)
    llm = HuggingFaceHub(repo_id="google/flan-t5-large", model_kwargs={"temperature": 0.3, "max_length": 512})
    return RetrievalQA.from_chain_type(llm=llm, retriever=db.as_retriever())


In [ ]:
# Gradio UI integration
qa_chain = None

def ingest(file):
    global qa_chain
    docs = load_documents(file)
    qa_chain = create_qa_chain(docs)
    return "Document ingested and model ready. Ask your questions."

def ask(q):
    if qa_chain:
        return qa_chain.run(q)
    return "Please ingest data first."

with gr.Blocks() as demo:
    gr.Markdown("# Real Estate Valuation QA")
    file_input = gr.File(label="Upload CSV or PDF")
    status = gr.Textbox(label="Status")
    ingest_btn = gr.Button("Ingest")
    ingest_btn.click(fn=ingest, inputs=file_input, outputs=status)
    
    question = gr.Textbox(label="Ask a question")
    answer = gr.Textbox(label="Answer")
    question.submit(fn=ask, inputs=question, outputs=answer)

demo.launch()